# Compass Dashboard

In [1]:
import pandas as pd

In [45]:
shift_data = {
    'month':['march','march','march'],
    'day':['Sunday','Monday','Tuesday'],
    'start':[1445,1445,1445],
    'end':[1835,1835,1835]
}

shifts = pd.DataFrame(shift_data)
shifts

,month,day,start,end
0,march,Sunday,1445,1835
1,march,Monday,1445,1835
2,march,Tuesday,1445,1835


In [46]:
fare_data = {
    '1zone': 2.70,
    '2zone': 4.00,
    '1zonemonth': 111.60,
    '2zonemonth': 149.25,
    '2zoneaddfare': 1.50,
    'yvraddfare' : 5.00
}
fare_data

{'1zone': 2.7,
 '2zone': 4.0,
 '1zonemonth': 111.6,
 '2zonemonth': 149.25,
 '2zoneaddfare': 1.5,
 'yvraddfare': 5.0}

In [48]:
def ShiftsToTrips(df:pd.DataFrame):
    "Convert a shifts schedule to the implied commute schedule"
    trips = df.copy()
    trips['start'] = df['start'] - 100
    trips.rename(columns={'start':'home','end':'YVR'}, inplace=True)
    trips = trips.melt(id_vars=['month','day'],
                        value_vars=['home','YVR'],
                        var_name='origin',
                        value_name='time')
    return trips

In [49]:
ShiftsToTrips(shifts)

,month,day,origin,time
0,march,Sunday,home,1345
1,march,Monday,home,1345
2,march,Tuesday,home,1345
3,march,Sunday,YVR,1835
4,march,Monday,YVR,1835
5,march,Tuesday,YVR,1835


In [ ]:
def GetOneZoneTrips(df:pd.DataFrame):
    "Find all trips that occur on a weekend or after 6:30pm"
    one_zone = df.loc[(df['day'].isin(['Saturday','Sunday'])) | (df['time']>1800)]

    return one_zone

GetOneZoneTrips(trips)

,month,day,origin,time
0,march,Sunday,home,1345
3,march,Sunday,YVR,1835
4,march,Monday,YVR,1835
5,march,Tuesday,YVR,1835


In [66]:
def GetTwoZoneTrips(df:pd.DataFrame):
    "find all trips that occur on a weekday before 6:30pm"
    two_zone = df.loc[(df['day'].isin(['Monday','Tuesday','Wednesday','Thursday','Friday']) & (df['time']<1800))]

    return two_zone

GetTwoZoneTrips(trips)

,month,day,origin,time
1,march,Monday,home,1345
2,march,Tuesday,home,1345


In [57]:
def GetAddFareTrips(df:pd.DataFrame):
    "find all trips that will incur the YVR add fare"
    add_fare = df.loc[df['origin'] == 'YVR']

    return add_fare

GetAddFareTrips(trips)

,month,day,origin,time
3,march,Sunday,YVR,1835
4,march,Monday,YVR,1835
5,march,Tuesday,YVR,1835


In [77]:
def TotalCost(shifts:pd.DataFrame, fares:dict, method:str):
    "find the total cost of commuting based on the given method"

    # convert shifts to trips
    trips = ShiftsToTrips(shifts)

    # count one zone trips
    OneZone = GetOneZoneTrips(trips).shape[0]

    # count two zone trips
    TwoZone = GetTwoZoneTrips(trips).shape[0]

    # count add fare trips
    AddFare = GetAddFareTrips(trips).shape[0]

    if method == 'StoredValue':
        TotalCost = OneZone*fares['1zone'] + TwoZone*fares['2zone'] + AddFare*fares['yvraddfare']
        return TotalCost
    if method == 'OneZoneMonthly':
        TotalCost = fares['1zonemonth'] + TwoZone*fares['2zoneaddfare']
        return TotalCost
    if method == 'TwoZoneMonthly':
        TotalCost = fares['2zonemonth']
        return TotalCost

    return None

TotalCost(shifts, fare_data, 'StoredValue')

33.8

In [78]:
CommuteCost = {
    'Stored Value' : TotalCost(shifts, fare_data, 'StoredValue'),
    '1 Zone Month Pass' : TotalCost(shifts, fare_data, 'OneZoneMonthly'),
    '2 Zone Month Pass' : TotalCost(shifts, fare_data, 'TwoZoneMonthly')
}

pd.DataFrame(CommuteCost, index=range(1))

,Stored Value,1 Zone Month Pass,2 Zone Month Pass
0,33.8,114.6,149.25


## Preprocessing


In [1]:
text = open("MAR 2026.txt").read()

In [2]:
text

'MAR 2026\nMon \n02-Mar \nTue \n03-Mar \nWed \n04-Mar \nThu \n05-Mar \nFri \n06-Mar \nSat \n07-Mar \nSun \n08-Mar\nRatna \nSalim\n\n\n1445-1845  \nAGT1\n\n\n\n\n1445-1845  \nAGT1\n\n\n\n\nAlex \nYe\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAnn \nKim\n\n\n1445-1845  \nAGT1\n1445-1845  \nAGT1\n\n\n1415-1815  \nTKT\n\n\n1545-1945  \nAGT1\nSaeko \nSumi\n\n\n1215-1915 FC \n1215-1915 FC \n1215-1915 FC \n1215-1915 FC \n1215-1915 FC\n\n\nJoey \nLu\n1115-1915  \nSUP\n1115-1915  \nSUP\n\n\n1115-1915  \nSUP\n\n\n1115-1915  \nSUP\n1215-2015  \nOPS\nCindy \nXiao\n\n\n\n\n1145-1845  \nARR+DEP\n\n\n1445-1845  \nAGT1\n\n\n1245-1945  \nARR+DEP\nSatomi \nHarward\n1145-1845  \nARR+DEP\n1145-1845  \nARR+DEP\n\n\n\n\n1145-1845  \nARR+DEP\n1145-1845  \nARR+DEP\n1245-1945  \nARR+DEP\nKohei \nTodoroki\n\n\n1415-1815  \nTKT\n\n\n1415-1815  \nTKT\n\n\n\n\n1515-1915  \nTKT\nRich \nHung\n\n\n\n\n\n\n1445-1845  \nAGT1\n\n\n1445-1845  \nAGT1\n\n\nCecillia  \nZhu\n\n\n\n\n1145-1845 WT 1145-1845 WT \n\n\n\n\n1145-1845 WT 1145-184

In [9]:
import csv
file_path = "MAR 2026.txt"

with open(file_path, mode='r') as file:
    reader = csv.reader(file, delimiter='\n')
    elements = [row for row in reader]

In [16]:
elements[15:31]

[['Ratna '],
 ['Salim'],
 [],
 [],
 ['1445-1845  '],
 ['AGT1'],
 [],
 [],
 [],
 [],
 ['1445-1845  '],
 ['AGT1'],
 [],
 [],
 [],
 []]

In [18]:
elements[15][0]+elements[16][0]

'Ratna Salim'

In [25]:
entries = []
left = 1
right = 2

while right < len(elements):
    entry = elements[left]+elements[right]
    if len(entry) >1:
         entry = entry[0]+entry[1]   
    entries.append(entry)
    left+=2
    right+=2


In [29]:
entries[7:15]


['Ratna Salim', [], '1445-1845  AGT1', [], [], '1445-1845  AGT1', [], []]

In [37]:
left=7
right=15
while right<len(entries):
    print(entries[left:right])
    left+=8
    right+=8

['Ratna Salim', [], '1445-1845  AGT1', [], [], '1445-1845  AGT1', [], []]
['Alex Ye', [], [], [], [], [], [], []]
['Ann Kim', [], '1445-1845  AGT1', '1445-1845  AGT1', [], '1415-1815  TKT', [], '1545-1945  AGT1']
['Saeko Sumi', [], '1215-1915 FC 1215-1915 FC ', '1215-1915 FC 1215-1915 FC ', ['1215-1915 FC'], ['Joey '], 'Lu1115-1915  ', 'SUP1115-1915  ']
[['SUP'], ['1115-1915  '], ['SUP'], ['1115-1915  '], 'SUP1215-2015  ', 'OPSCindy ', ['Xiao'], []]
[['1145-1845  '], ['ARR+DEP'], ['1445-1845  '], ['AGT1'], ['1245-1945  '], 'ARR+DEPSatomi ', 'Harward1145-1845  ', 'ARR+DEP1145-1845  ']
[['ARR+DEP'], [], ['1145-1845  '], 'ARR+DEP1145-1845  ', 'ARR+DEP1245-1945  ', 'ARR+DEPKohei ', ['Todoroki'], ['1415-1815  ']]
[['TKT'], ['1415-1815  '], ['TKT'], [], ['1515-1915  '], 'TKTRich ', ['Hung'], []]
[[], ['1445-1845  '], ['AGT1'], ['1445-1845  '], ['AGT1'], ['Cecillia  '], ['Zhu'], []]
[['1145-1845 WT 1145-1845 WT '], [], [], '1145-1845 WT 1145-1845 WT1245-1945 WT', 'Donia  Forbes', '1445-1845  

In [6]:
import pdfplumber

with pdfplumber.open("Monthly Schedule _ MAR.pdf") as pdf:
    page = pdf.pages[0]
    table = page.extract_table()

print(table)

[['MAR 2026', 'Mon\n02-Mar', 'Tue\n03-Mar', 'Wed\n04-Mar', 'Thu\n05-Mar', 'Fri\n06-Mar', 'Sat\n07-Mar', 'Sun\n08-Mar'], ['Ratna\nSalim', '', '1445-1845\nAGT1', '', '', '1445-1845\nAGT1', '', ''], ['Alex\nYe', '', '', '', '', '', '', ''], ['Ann\nKim', '', '1445-1845\nAGT1', '1445-1845\nAGT1', '', '1415-1815\nTKT', '', '1545-1945\nAGT1'], ['Saeko\nSumi', '', '1215-1915 FC', '1215-1915 FC', '1215-1915 FC', '1215-1915 FC', '1215-1915 FC', ''], ['Joey\nLu', '1115-1915\nSUP', '1115-1915\nSUP', '', '1115-1915\nSUP', '', '1115-1915\nSUP', '1215-2015\nOPS'], ['Cindy\nXiao', '', '', '1145-1845\nARR+DEP', '', '1445-1845\nAGT1', '', '1245-1945\nARR+DEP'], ['Satomi\nHarward', '1145-1845\nARR+DEP', '1145-1845\nARR+DEP', '', '', '1145-1845\nARR+DEP', '1145-1845\nARR+DEP', '1245-1945\nARR+DEP'], ['Kohei\nTodoroki', '', '1415-1815\nTKT', '', '1415-1815\nTKT', '', '', '1515-1915\nTKT'], ['Rich\nHung', '', '', '', '1445-1845\nAGT1', '', '1445-1845\nAGT1', ''], ['Cecillia\nZhu', '', '1145-1845 WT', '1145-

In [7]:
pd.DataFrame(table)

,0,1,2,3,4,5,6,7
0,MAR 2026,Mon\n02-Mar,Tue\n03-Mar,Wed\n04-Mar,Thu\n05-Mar,Fri\n06-Mar,Sat\n07-Mar,Sun\n08-Mar
1,Ratna\nSalim,,1445-1845\nAGT1,,,1445-1845\nAGT1,,
2,Alex\nYe,,,,,,,
3,Ann\nKim,,1445-1845\nAGT1,1445-1845\nAGT1,,1415-1815\nTKT,,1545-1945\nAGT1
4,Saeko\nSumi,,1215-1915 FC,1215-1915 FC,1215-1915 FC,1215-1915 FC,1215-1915 FC,
5,Joey\nLu,1115-1915\nSUP,1115-1915\nSUP,,1115-1915\nSUP,,1115-1915\nSUP,1215-2015\nOPS
6,Cindy\nXiao,,,1145-1845\nARR+DEP,,1445-1845\nAGT1,,1245-1945\nARR+DEP
7,Satomi\nHarward,1145-1845\nARR+DEP,1145-1845\nARR+DEP,,,1145-1845\nARR+DEP,1145-1845\nARR+DEP,1245-1945\nARR+DEP
8,Kohei\nTodoroki,,1415-1815\nTKT,,1415-1815\nTKT,,,1515-1915\nTKT
9,Rich\nHung,,,,1445-1845\nAGT1,,1445-1845\nAGT1,


In [8]:
with pdfplumber.open("Monthly Schedule _ MAR.pdf") as pdf:
    page = pdf.pages[0]
    text = page.extract_text()

print(text)

Mon Tue Wed Thu Fri Sat Sun
MAR 2026
02-Mar 03-Mar 04-Mar 05-Mar 06-Mar 07-Mar 08-Mar
Ratna 1445-1845 1445-1845
Salim AGT1 AGT1
Alex
Ye
Ann 1445-1845 1445-1845 1415-1815 1545-1945
Kim AGT1 AGT1 TKT AGT1
Saeko
1215-1915 FC 1215-1915 FC 1215-1915 FC 1215-1915 FC 1215-1915 FC
Sumi
Joey 1115-1915 1115-1915 1115-1915 1115-1915 1215-2015
Lu SUP SUP SUP SUP OPS
Cindy 1145-1845 1445-1845 1245-1945
Xiao ARR+DEP AGT1 ARR+DEP
Satomi 1145-1845 1145-1845 1145-1845 1145-1845 1245-1945
Harward ARR+DEP ARR+DEP ARR+DEP ARR+DEP ARR+DEP
Kohei 1415-1815 1415-1815 1515-1915
Todoroki TKT TKT TKT
Rich 1445-1845 1445-1845
Hung AGT1 AGT1
Cecillia
1145-1845 WT 1145-1845 WT 1145-1845 WT 1145-1845 WT 1245-1945 WT
Zhu
Donia 1445-1845 1445-1845 1445-1845
Forbes AGT1 AGT1 AGT1
Kevin 1145-1845 1145-1845
Lee ARR+DEP ARR+DEP
Crystal 1145-1845 1145-1845 1145-1845
1215-1915 FC 1315-2015 FC
Deng ARR+DEP ARR+DEP ARR+DEP
Eunsol 1445-1845 1445-1845 1445-1845 1445-1845
Choi AGT1 AGT1 AGT1 AGT1
Angel 1415-1815 1145-1815 1145-1

In [9]:
with pdfplumber.open("Monthly Schedule _ MAR.pdf") as pdf:
    page = pdf.pages[0]
    words = page.extract_words()

for w in words[:20]:
    print(w)

{'text': 'Mon', 'x0': 103.44, 'x1': 123.0912, 'top': 57.593279999999936, 'doctop': 57.593279999999936, 'bottom': 68.63328000000001, 'upright': True, 'height': 11.040000000000077, 'width': 19.651200000000003, 'direction': 'ltr'}
{'text': 'Tue', 'x0': 177.26, 'x1': 193.62127999999998, 'top': 57.593279999999936, 'doctop': 57.593279999999936, 'bottom': 68.63328000000001, 'upright': True, 'height': 11.040000000000077, 'width': 16.361279999999994, 'direction': 'ltr'}
{'text': 'Wed', 'x0': 247.13, 'x1': 267.97216, 'top': 57.593279999999936, 'doctop': 57.593279999999936, 'bottom': 68.63328000000001, 'upright': True, 'height': 11.040000000000077, 'width': 20.84215999999998, 'direction': 'ltr'}
{'text': 'Thu', 'x0': 321.43, 'x1': 337.92375999999996, 'top': 57.593279999999936, 'doctop': 57.593279999999936, 'bottom': 68.63328000000001, 'upright': True, 'height': 11.040000000000077, 'width': 16.493759999999952, 'direction': 'ltr'}
{'text': 'Fri', 'x0': 396.19, 'x1': 407.6164, 'top': 57.593279999999